# Comparaison systématique : 4 modèles x 3 stratégies de déséquilibre

Module 2 du sujet (Khady KAMA).
 
Modèles (>= 4 exigés) :
  - Logistic Regression (baseline)
  - Random Forest
  - XGBoost
  - LightGBM
 
Stratégies de déséquilibre (>= 3 exigées) :
  - Sous-échantillonnage aléatoire (RandomUnderSampler)
  - SMOTE (suréchantillonnage synthétique)
  - Coût pondéré (class_weight / scale_pos_weight, sans rééchantillonnage)
 
Métrique principale : AUC-PR (imposée par le sujet, plus adaptée que
l'accuracy sur données déséquilibrées).
Métriques complémentaires : F1, précision, rappel, AUC-ROC, temps
d'entraînement, temps d'inférence.
 
Prérequis : avoir exécuté split_train_val_test.py au préalable
(fichiers X_train.csv, X_val.csv, y_train.csv, y_val.csv).
 
Installation si besoin :
  pip install imbalanced-learn xgboost lightgbm
 
Auteur : Rasmané

In [ ]:
import pandas as pd
import numpy as np
import time
import warnings
 
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    average_precision_score, f1_score, precision_score,
    recall_score, roc_auc_score, confusion_matrix
)
 
from imblearn.under_sampling import RandomUnderSampler
from imblearn.over_sampling import SMOTE
 
import xgboost as xgb
import lightgbm as lgb
 
warnings.filterwarnings("ignore")
 
SEED = 42
np.random.seed(SEED)
 
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 150)

## 0. CHARGEMENT DES DONNÉES SPLITTÉES

In [ ]:
DOSSIER = r"C:\Users\hp\Documents\Fraude_detection\data"
 
X_train = pd.read_csv(f"{DOSSIER}/X_train.csv")
X_val = pd.read_csv(f"{DOSSIER}/X_val.csv")
y_train = pd.read_csv(f"{DOSSIER}/y_train.csv").squeeze()
y_val = pd.read_csv(f"{DOSSIER}/y_val.csv").squeeze()
 
print(f"X_train : {X_train.shape} | Taux de fraude : {y_train.mean()*100:.4f}%")
print(f"X_val   : {X_val.shape} | Taux de fraude : {y_val.mean()*100:.4f}%")
 
# Sécurité : s'assurer qu'il n'y a que des colonnes numériques
# (le one-hot encoding du type a déjà dû être fait à l'étape précédente)
colonnes_non_numeriques = X_train.select_dtypes(exclude=[np.number]).columns.tolist()
if colonnes_non_numeriques:
    print(f"\nATTENTION : colonnes non numériques détectées : {colonnes_non_numeriques}")
    print("Elles doivent être encodées avant modélisation. Arrêt du script.")
    raise ValueError("Colonnes non numériques présentes dans X_train.")

## 1. DÉFINITION DES STRATÉGIES DE GESTION DU DÉSÉQUILIBRE

In [ ]:
def preparer_sous_echantillonnage(X, y):
    """Sous-échantillonnage aléatoire de la classe majoritaire."""
    sampler = RandomUnderSampler(random_state=SEED)
    X_res, y_res = sampler.fit_resample(X, y)
    return X_res, y_res
 
 
def preparer_smote(X, y):
    """Suréchantillonnage synthétique de la classe minoritaire (SMOTE)."""
    sampler = SMOTE(random_state=SEED)
    X_res, y_res = sampler.fit_resample(X, y)
    return X_res, y_res
 
 
def preparer_cout_pondere(X, y):
    """
    Pas de rééchantillonnage : les données restent inchangées.
    Le déséquilibre est géré directement dans les hyperparamètres
    du modèle (class_weight, scale_pos_weight, is_unbalance).
    """
    return X, y
 
 
STRATEGIES = {
    "sous_echantillonnage": preparer_sous_echantillonnage,
    "smote": preparer_smote,
    "cout_pondere": preparer_cout_pondere,
}

## 2. DÉFINITION DES MODÈLES

In [ ]:
# Le paramètre de gestion du déséquilibre n'est activé QUE pour la
# stratégie "cout_pondere". Pour les deux autres stratégies (données déjà
# rééquilibrées par rééchantillonnage), on utilise les réglages par défaut
# pour ne pas cumuler deux mécanismes de correction du déséquilibre.
 
def creer_modeles(strategie, ratio_desequilibre):
    """
    Retourne un dictionnaire de modèles non entraînés, configurés selon
    la stratégie en cours.
    ratio_desequilibre = nb_non_fraude / nb_fraude, utilisé pour
    scale_pos_weight (XGBoost/LightGBM).
    """
    utiliser_poids = (strategie == "cout_pondere")
 
    modeles = {
        "LogisticRegression": LogisticRegression(
            max_iter=1000,
            random_state=SEED,
            class_weight="balanced" if utiliser_poids else None,
            n_jobs=-1,
        ),
        "RandomForest": RandomForestClassifier(
            n_estimators=200,
            max_depth=12,
            random_state=SEED,
            class_weight="balanced" if utiliser_poids else None,
            n_jobs=-1,
        ),
        "XGBoost": xgb.XGBClassifier(
            n_estimators=300,
            max_depth=6,
            learning_rate=0.1,
            scale_pos_weight=ratio_desequilibre if utiliser_poids else 1,
            random_state=SEED,
            eval_metric="aucpr",
            n_jobs=-1,
            verbosity=0,
        ),
        "LightGBM": lgb.LGBMClassifier(
            n_estimators=300,
            max_depth=6,
            learning_rate=0.1,
            is_unbalance=utiliser_poids,
            random_state=SEED,
            n_jobs=-1,
            verbosity=-1,
        ),
    }
    return modeles

## 3. BOUCLE DE COMPARAISON SYSTÉMATIQUE

In [ ]:
resultats = []
 
for nom_strategie, fonction_strategie in STRATEGIES.items():
    print(f"\n{'='*70}")
    print(f"STRATÉGIE : {nom_strategie}")
    print(f"{'='*70}")
 
    debut_prep = time.time()
    X_train_strat, y_train_strat = fonction_strategie(X_train, y_train)
    duree_prep = time.time() - debut_prep
 
    print(f"Préparation des données : {duree_prep:.1f}s")
    print(f"Taille après stratégie : {X_train_strat.shape[0]:,} lignes")
    print(f"Taux de fraude après stratégie : {y_train_strat.mean()*100:.2f}%")
 
    ratio_desequilibre = (y_train == 0).sum() / (y_train == 1).sum()
    modeles = creer_modeles(nom_strategie, ratio_desequilibre)
 
    for nom_modele, modele in modeles.items():
        print(f"\n--- {nom_modele} | {nom_strategie} ---")
 
        # Entraînement
        debut_train = time.time()
        modele.fit(X_train_strat, y_train_strat)
        duree_train = time.time() - debut_train
 
        # Inférence sur validation
        debut_inference = time.time()
        y_proba = modele.predict_proba(X_val)[:, 1]
        duree_inference = time.time() - debut_inference
        temps_inference_par_transaction_ms = (duree_inference / len(X_val)) * 1000
 
        y_pred = (y_proba >= 0.5).astype(int)  # seuil par défaut 0.5, optimisé plus tard
 
        # Calcul des métriques
        auc_pr = average_precision_score(y_val, y_proba)
        auc_roc = roc_auc_score(y_val, y_proba)
        f1 = f1_score(y_val, y_pred)
        precision = precision_score(y_val, y_pred, zero_division=0)
        rappel = recall_score(y_val, y_pred)
 
        tn, fp, fn, tp = confusion_matrix(y_val, y_pred).ravel()
 
        print(f"AUC-PR : {auc_pr:.4f} | AUC-ROC : {auc_roc:.4f} | "
              f"F1 : {f1:.4f} | Precision : {precision:.4f} | Recall : {rappel:.4f}")
        print(f"Temps entraînement : {duree_train:.1f}s | "
              f"Temps inférence/transaction : {temps_inference_par_transaction_ms:.4f}ms")
        print(f"Matrice de confusion -> VP={tp} FP={fp} FN={fn} VN={tn}")
 
        resultats.append({
            "strategie": nom_strategie,
            "modele": nom_modele,
            "auc_pr": round(auc_pr, 4),
            "auc_roc": round(auc_roc, 4),
            "f1_score": round(f1, 4),
            "precision": round(precision, 4),
            "recall": round(rappel, 4),
            "temps_entrainement_s": round(duree_train, 1),
            "temps_inference_ms_par_transaction": round(temps_inference_par_transaction_ms, 4),
            "vrais_positifs": int(tp),
            "faux_positifs": int(fp),
            "faux_negatifs": int(fn),
            "vrais_negatifs": int(tn),
        })
 
    # Libération mémoire avant la stratégie suivante
    del X_train_strat, y_train_strat
    import gc
    gc.collect()

## 4. TABLEAU COMPARATIF FINAL

In [ ]:
df_resultats = pd.DataFrame(resultats)
df_resultats = df_resultats.sort_values("auc_pr", ascending=False).reset_index(drop=True)
 
print(f"\n{'='*70}")
print("TABLEAU COMPARATIF COMPLET (trié par AUC-PR décroissant)")
print(f"{'='*70}")
print(df_resultats.to_string(index=False))

## 5. IDENTIFICATION DU MEILLEUR MODÈLE

In [ ]:
meilleure_ligne = df_resultats.iloc[0]
print(f"\n{'='*70}")
print("MEILLEURE COMBINAISON (selon AUC-PR)")
print(f"{'='*70}")
print(f"Modèle    : {meilleure_ligne['modele']}")
print(f"Stratégie : {meilleure_ligne['strategie']}")
print(f"AUC-PR    : {meilleure_ligne['auc_pr']}")
print(f"Recall    : {meilleure_ligne['recall']}")
 
print("\nSeuils de performance attendus par le sujet :")
print("- Logistic Regression (baseline) : AUC-PR >= 0.60")
print("- Meilleur modèle (XGBoost/LightGBM) : AUC-PR >= 0.90, Recall >= 0.85")
 
baseline_lr = df_resultats[df_resultats["modele"] == "LogisticRegression"]["auc_pr"].max()
print(f"\nMeilleur AUC-PR obtenu par Logistic Regression : {baseline_lr:.4f} "
      f"-> {'OK' if baseline_lr >= 0.60 else 'EN-DESSOUS DU SEUIL'}")
print(f"Meilleur AUC-PR global : {meilleure_ligne['auc_pr']:.4f} "
      f"-> {'OK' if meilleure_ligne['auc_pr'] >= 0.90 else 'EN-DESSOUS DU SEUIL'}")
print(f"Recall associé : {meilleure_ligne['recall']:.4f} "
      f"-> {'OK' if meilleure_ligne['recall'] >= 0.85 else 'EN-DESSOUS DU SEUIL'}")

## 6. TABLEAU PIVOT POUR LE RAPPORT (modèle x stratégie, AUC-PR)

In [ ]:
pivot_auc_pr = df_resultats.pivot(index="modele", columns="strategie", values="auc_pr")
print(f"\n{'='*70}")
print("TABLEAU PIVOT - AUC-PR par modèle x stratégie (pour le rapport)")
print(f"{'='*70}")
print(pivot_auc_pr)

## 7. SAUVEGARDE DES RÉSULTATS

In [ ]:
CHEMIN_RESULTATS = f"{DOSSIER}/resultats_comparaison_modeles.csv"
df_resultats.to_csv(CHEMIN_RESULTATS, index=False)
print(f"\nTableau complet sauvegardé : {CHEMIN_RESULTATS}")
 
print("""
NOTE MÉTHODOLOGIQUE (à reprendre dans le rapport) :
Quatre modèles (Logistic Regression, Random Forest, XGBoost, LightGBM)
ont été comparés selon trois stratégies de gestion du déséquilibre de
classes (sous-échantillonnage aléatoire, SMOTE, coût pondéré), soit 12
combinaisons évaluées sur l'ensemble de validation. La métrique
principale retenue est l'AUC-PR, plus adaptée que l'accuracy ou l'AUC-ROC
sur des données fortement déséquilibrées car elle se concentre sur la
performance de détection de la classe minoritaire (fraude). Le seuil de
décision de 0.5 utilisé ici est provisoire ; son optimisation via la
courbe Precision-Recall sera effectuée à l'étape suivante sur le modèle
retenu.
""")